<a href="https://colab.research.google.com/github/Trannganne/face-mask-tracking-system/blob/feature%2Ftracking_timer/training_updated.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Chạy trong Google Colab:



In [1]:
# Mount drive và giải nén
from google.colab import drive
drive.mount('/content/drive')
!unzip -oq "/content/drive/MyDrive/Face_Mask_Project/Dataset_mask.zip" -d /content/dataset

Mounted at /content/drive


In [ ]:
!pip install torch torchvision opencv-python pillow matplotlib seaborn scikit-learn
!pip install deep-sort-realtime
!apt-get install -y libsndfile1
!pip install soundfile
!pip install facenet-pytorch  # Để detect face

In [ ]:
# Xử lý xung đột version
!pip install "numpy>=2.0" "torch>=2.4" "torchvision>=0.19" facenet-pytorch

In [27]:
#Kiểm tra các phiên bản
import numpy as np
import torch
import torchvision
from facenet_pytorch import MTCNN, InceptionResnetV1
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_recall_fscore_support
)

print(f"NumPy: {np.__version__}")
print(f"Torch: {torch.__version__}")
print(f"TorchVision: {torchvision.__version__}")
print(f"GPU Available: {torch.cuda.is_available()}")

# Chạy thử một lệnh của facenet-pytorch
try:
    mtcnn = MTCNN()
    print("facenet-pytorch hoạt động bình thường trên NumPy 2.x!")
except Exception as e:
    print(f"Có lỗi phát sinh: {e}")

NumPy: 2.4.6
Torch: 2.12.0+cu130
TorchVision: 0.27.0+cu130
GPU Available: True
facenet-pytorch hoạt động bình thường trên NumPy 2.x!


In [2]:
from facenet_pytorch import InceptionResnetV1

model = InceptionResnetV1(pretrained='vggface2')
print("Load model OK")

  0%|          | 0.00/107M [00:00<?, ?B/s]

Load model OK


In [ ]:
import torch
t = torch.cuda.get_device_properties(0)
print(f"Tổng bộ nhớ GPU: {t.total_memory / 1024**2:.2f} MB")

In [28]:
# ============================================================
# TRANSFER LEARNING - MobileNetV2 (AN TOÀN CHO ĐỒ ÁN)
# ============================================================

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models

class MaskDetectionTransfer(nn.Module):

    def __init__(self, num_classes=3):
        super(MaskDetectionTransfer, self).__init__()

        # Load MobileNetV2 pretrained
        self.model = models.mobilenet_v2(pretrained=True)

        # Freeze backbone
        for param in self.model.features.parameters():
            param.requires_grad = False

        # Thay classifier cuối
        in_features = self.model.classifier[1].in_features

        self.model.classifier = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.model(x)

 KIẾN TRÚC MODEL: Custom CNN
    
    INPUT: (batch, 3, 128, 128) - Ảnh RGB 128x128
    
    BLOCK 1: Convolutional Block 1
    - Conv2d: 3 → 32 channels, kernel=3, padding=1
    - BatchNorm2d: 32 channels
    - ReLU activation
    - MaxPool2d: kernel=2, stride=2
    - Dropout: p=0.25
    → Output: (batch, 32, 64, 64)
    BLOCK 2: Convolutional Block 2
    - Conv2d: 32 → 64 channels, kernel=3, padding=1
    - BatchNorm2d: 64 channels
    - ReLU activation
    - MaxPool2d: kernel=2, stride=2
    - Dropout: p=0.25
    → Output: (batch, 64, 32, 32)
    
    BLOCK 3: Convolutional Block 3
    - Conv2d: 64 → 128 channels, kernel=3, padding=1
    - BatchNorm2d: 128 channels
    - ReLU activation
    - MaxPool2d: kernel=2, stride=2
    - Dropout: p=0.3
    → Output: (batch, 128, 16, 16)
    
    BLOCK 4: Convolutional Block 4
    - Conv2d: 128 → 256 channels, kernel=3, padding=1
    - BatchNorm2d: 256 channels
    - ReLU activation
    - MaxPool2d: kernel=2, stride=2
    - Dropout: p=0.3
    → Output: (batch, 256, 8, 8)
    FLATTEN: (batch, 256, 8, 8) → (batch, 16384)
    
    FULLY CONNECTED LAYERS:
    - FC1: 16384 → 512 nodes
      - ReLU activation
      - Dropout: p=0.5
    
    - FC2: 512 → 128 nodes
      - ReLU activation
      - Dropout: p=0.5
    
    - FC3 (Output): 128 → 3 nodes (3 classes)
    
    OUTPUT: (batch, 3) - Logits cho 3 classes:
      0: Đeo sai cách (incorrect_mask)
      1: Đeo đúng cách (with_mask)
      2: Không đeo (without_mask)
    
    TỔNG SỐ THAM SỐ: ~8.7 triệu parameters

============= Bước 3: HÀM TỐI ƯU HÓA ====================

 HÀM TRAIN MODEL
    
    Optimizer: Adam
    - Learning rate: 0.001
    - Weight decay: 1e-4 (L2 regularization)
    
    Loss Function: CrossEntropyLoss
    
    Learning Rate Scheduler: ReduceLROnPlateau
    - Giảm LR khi val_loss không giảm sau 5 epochs
    - Factor: 0.5 (giảm 50%)
    
    Early Stopping:
    - Dừng nếu val_loss không cải thiện sau 10 epochs

In [29]:
# ============================================================
# TRAIN MODEL - SO SÁNH ADAM VÀ SGD
# ============================================================

import torch
import torch.nn as nn
import torch.optim as optim
import time
import json

def train_model(
    model,
    train_loader,
    val_loader,
    num_epochs=10,
    optimizer_name="adam"
):

    device = torch.device(
        'cuda' if torch.cuda.is_available() else 'cpu'
    )

    model.to(device)

    # ========================================================
    # LOSS FUNCTION
    # ========================================================
    criterion = nn.CrossEntropyLoss()

    # ========================================================
    # CHỌN OPTIMIZER
    # ========================================================
    if optimizer_name.lower() == "adam":

        optimizer = optim.Adam(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=0.001,
            weight_decay=1e-4
        )

        print("\nĐang sử dụng optimizer: ADAM")

    elif optimizer_name.lower() == "sgd":

        optimizer = optim.SGD(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=0.01,
            momentum=0.9,
            weight_decay=1e-4
        )

        print("\nĐang sử dụng optimizer: SGD + Momentum")

    else:
        raise ValueError("Optimizer không hợp lệ!")

    # ========================================================
    # LEARNING RATE SCHEDULER
    # ========================================================
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=0.5,
        patience=2
    )

    # ========================================================
    # EARLY STOPPING
    # ========================================================
    best_val_loss = float('inf')

    patience = 5

    patience_counter = 0

    # ========================================================
    # HISTORY
    # ========================================================
    history = {
        'train_loss': [],
        'val_loss': [],
        'train_acc': [],
        'val_acc': []
    }

    # ========================================================
    # TRAINING LOOP
    # ========================================================
    for epoch in range(num_epochs):

        start_time = time.time()

        # ====================================================
        # TRAIN
        # ====================================================
        model.train()

        running_loss = 0.0

        train_correct = 0

        train_total = 0

        for images, labels in train_loader:

            images = images.to(device)

            labels = labels.to(device)

            # Forward
            outputs = model(images)

            loss = criterion(outputs, labels)

            # Backward
            optimizer.zero_grad()

            loss.backward()

            optimizer.step()

            # Statistics
            running_loss += loss.item()

            _, predicted = torch.max(outputs, 1)

            train_total += labels.size(0)

            train_correct += (predicted == labels).sum().item()

        train_loss = running_loss / len(train_loader)

        train_acc = 100 * train_correct / train_total

        # ====================================================
        # VALIDATION
        # ====================================================
        model.eval()

        val_loss = 0.0

        val_correct = 0

        val_total = 0

        with torch.no_grad():

            for images, labels in val_loader:

                images = images.to(device)

                labels = labels.to(device)

                outputs = model(images)

                loss = criterion(outputs, labels)

                val_loss += loss.item()

                _, predicted = torch.max(outputs, 1)

                val_total += labels.size(0)

                val_correct += (predicted == labels).sum().item()

        val_loss = val_loss / len(val_loader)

        val_acc = 100 * val_correct / val_total

        # ====================================================
        # UPDATE LEARNING RATE
        # ====================================================
        scheduler.step(val_loss)

        # ====================================================
        # SAVE HISTORY
        # ====================================================
        history['train_loss'].append(train_loss)

        history['val_loss'].append(val_loss)

        history['train_acc'].append(train_acc)

        history['val_acc'].append(val_acc)

        # ====================================================
        # THỜI GIAN TRAIN
        # ====================================================
        epoch_time = time.time() - start_time

        # ====================================================
        # OUTPUT KIỂU KERAS
        # ====================================================
        print(
            f"\nEpoch {epoch+1}/{num_epochs}"
        )

        print(
            f"{len(train_loader)}/{len(train_loader)} "
            f"━━━━━━━━━━━━━━━━━━━━ "
            f"{epoch_time:.0f}s "
            f"- accuracy: {train_acc/100:.4f} "
            f"- loss: {train_loss:.4f} "
            f"- val_accuracy: {val_acc/100:.4f} "
            f"- val_loss: {val_loss:.4f}"
        )

        # ====================================================
        # EARLY STOPPING
        # ====================================================
        if val_loss < best_val_loss:

            best_val_loss = val_loss

            patience_counter = 0

            torch.save(
                model.state_dict(),
                f'/content/drive/MyDrive/best_model_{optimizer_name}.pth'
            )

            print("Đã lưu model tốt nhất!")

        else:

            patience_counter += 1

            print(f"Không cải thiện ({patience_counter}/{patience})")

            if patience_counter >= patience:

                print(f"\nDừng sớm tại epoch {epoch+1}")

                break

    save_path = f"/content/drive/MyDrive/history_{optimizer_name}.json"

    with open(save_path, "w") as f:
        json.dump(history, f)

    return model, history

BƯỚC 4: HÀM ĐÁNH GIÁ

 Metrics:
    
    - Precision: TP / (TP + FP)
    
    - Recall: TP / (TP + FN)
    
    - F1-Score: 2 * (Precision * Recall) / (Precision + Recall)
    
    - Confusion Matrix


In [30]:
from sklearn.metrics import confusion_matrix, classification_report, precision_recall_fscore_support

def evaluate_model(model, test_loader, class_names=['Đeo sai', 'Đeo đúng', 'Không đeo']):
  device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
  model.to(device)
  model.eval()

  all_preds = []
  all_labels = []

  with torch.no_grad():
      for images, labels in test_loader:
          images = images.to(device)
          outputs = model(images)
          _, predicted = torch.max(outputs.data, 1)

          all_preds.extend(predicted.cpu().numpy())
          all_labels.extend(labels.numpy())

  # Calculate metrics
  precision, recall, f1, support = precision_recall_fscore_support(
      all_labels, all_preds, average=None
    )

  # IN metrics
  print("\n"+"="*60)
  print("Đánh giá model")
  print("="*60)

  for i, class_name in enumerate(class_names):
    print(f"{class_name}:")
    print(f"Precision: {precision[i]:.4f}")
    print(f"Recall: {recall[i]:.4f}")
    print(f"F1-Score: {f1[i]:.4f}")
    print(f"Support: {support[i]}")

  # Metrics tổng quan
  overall_precision = np.mean(precision)
  overall_recall = np.mean(recall)
  overall_f1 = np.mean(f1)

  print("\n"+"="*60)
  print(f"Precision: {overall_precision:.4f}")
  print(f"Recall: {overall_recall:.4f}")
  print(f"F1-Score: {overall_f1:.4f}")
  print("="*60)

  # Confusion matrix
  cm = confusion_matrix(all_labels, all_preds)
  print("Confusion Matrix:")
  print(cm)

  # Báo cáo phân loại
  print("\n Báo cáo")
  print(classification_report(all_labels, all_preds, target_names=class_names))

  # Ma trận nhầm lẫn
  cm=confusion_matrix(all_labels, all_preds)

  plt.figure(figsize=(10,8))
  sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
  plt.title('Confusion Matrix', fontsize=16, fontweight='bold')
  plt.xlabel('Predicted Label', fontsize=12)
  plt.ylabel('True Label', fontsize=12)
  plt.tight_layout()
  plt.savefig('confusion_matrix.png',dpi=300, bbox_inches='tight')
  plt.show()

  return precision, recall, f1

BƯỚC 5: DEEPSORT TRACKING + CẢNH BÁO
Chức năng:
    - Track từng người bằng DeepSORT
    - Đếm thời gian không đeo khẩu trang
    - Phát cảnh báo nếu > 20s

---
## ⏱️ Cập nhật: PersonTrackerV2 (Logic đúng + 10s)


In [16]:
# ============================================================
# ⏱️ TRACKER CẬP NHẬT: đếm thời gian ĐEO KHẨU TRANG
#    - Đeo vào (with_mask)  → bắt đầu đếm
#    - Tháo ra / đeo sai    → reset về 0
#    - Vượt 10 giây         → CẢnh BÁO ĐỎ
# ============================================================
from collections import defaultdict
import time

class PersonTrackerV2:
    """
    DeepSORT tracker kết hợp đếm thời gian đeo khẩu trang.

    CLASS IDs (theo ImageFolder thứ tự alphabet):
        0 = mask_weared_incorrect  (đeo sai)
        1 = with_mask              (đeo đúng)
        2 = without_mask           (không đeo)

    Logic cảnh báo:
        - Nếu class == 1 (đeo đúng): bắt đầu / tiếp tục đếm
        - Nếu class != 1            : reset về 0
        - Nếu thời gian đeo > ALERT_SECONDS: hiện CẢNH BÁO
    """
    ALERT_SECONDS = 10          # ← Ngưỡng cảnh báo (giây)

    def __init__(self):
        from deep_sort_realtime.deepsort_tracker import DeepSort
        self.tracker = DeepSort(
            max_age=30,
            n_init=3,
            nn_budget=100,
            max_iou_distance=0.7
        )
        # Trạng thái mỗi track_id
        self.states = defaultdict(lambda: {
            "mask_start": None,      # thời điểm bắt đầu đeo
            "mask_duration": 0.0,    # tổng thời gian đang đeo
            "last_class": None,
            "alerted": False,
        })

    # ── Cập nhật mỗi frame ──────────────────────────────────
    def update(self, detections, frame_rgb, current_time):
        """
        detections: list of (bbox_xyxy, confidence, class_id)
        frame_rgb:  numpy array (H,W,3) — cần cho DeepSORT appearance
        current_time: float (giây từ đầu video)
        Returns: list of result dicts
        """
        ds_input = []
        for (x1, y1, x2, y2), conf, cls in detections:
            ds_input.append(([x1, y1, x2 - x1, y2 - y1], conf, cls))

        tracks = self.tracker.update_tracks(ds_input, frame=frame_rgb)
        results = []

        for track in tracks:
            if not track.is_confirmed():
                continue

            tid    = track.track_id
            bbox   = track.to_ltrb()          # (x1,y1,x2,y2)
            cls_id = track.get_det_class()
            st     = self.states[tid]

            # ── Logic đếm thời gian ──────────────────────
            if cls_id == 1:                    # đang đeo đúng
                if st["mask_start"] is None:
                    st["mask_start"] = current_time   # bắt đầu đếm
                st["mask_duration"] = current_time - st["mask_start"]
            else:                              # tháo ra / đeo sai
                st["mask_start"]    = None
                st["mask_duration"] = 0.0
                st["alerted"]       = False   # reset cảnh báo

            # ── Kiểm tra ngưỡng ──────────────────────────
            alert = (cls_id == 1 and st["mask_duration"] > self.ALERT_SECONDS)
            if alert and not st["alerted"]:
                print(f"CẢNH BÁO! ID {tid} đeo khẩu trang "
                      f"{st['mask_duration']:.1f}s > {self.ALERT_SECONDS}s")
                st["alerted"] = True

            st["last_class"] = cls_id
            results.append({
                "track_id":      tid,
                "bbox":          bbox,
                "class_id":      cls_id,
                "mask_duration": st["mask_duration"],
                "alert":         alert,
            })

        return results

    # ── Vẽ kết quả lên frame ────────────────────────────────
    @staticmethod
    def draw(frame, results, class_names):
        COLOR_MAP = {
            0: (0, 165, 255),   # đeo sai  → cam
            1: (0, 255, 0),     # đeo đúng → xanh lá
            2: (0, 0, 255),     # không đeo → đỏ
        }
        for r in results:
            x1, y1, x2, y2 = map(int, r["bbox"])
            cls   = r["class_id"]
            dur   = r["mask_duration"]
            alert = r["alert"]
            color = COLOR_MAP.get(cls, (200, 200, 200))

            # Bounding box (đậm hơn khi cảnh báo)
            thickness = 4 if alert else 2
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, thickness)

            # Nhãn + ID
            label = f"ID:{r['track_id']} {class_names[cls]}"
            cv2.putText(frame, label, (x1, y1 - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

            # Thời gian đeo (chỉ hiện khi đang đeo)
            if cls == 1:
                time_txt = f"Thoi gian deo: {dur:.1f}s"
                cv2.putText(frame, time_txt, (x1, y2 + 20),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 0), 2)

            # Banner cảnh báo đỏ
            if alert:
                overlay = frame.copy()
                cv2.rectangle(overlay, (x1, y1 - 55), (x2, y1 - 12),
                              (0, 0, 220), -1)
                cv2.addWeighted(overlay, 0.6, frame, 0.4, 0, frame)
                cv2.putText(frame,
                            f"!! CANH BAO: {dur:.0f}s !!",
                            (x1 + 4, y1 - 22),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 255), 2)
        return frame

print(f"PersonTrackerV2 sẵn sàng — ngưỡng cảnh báo: {PersonTrackerV2.ALERT_SECONDS}s")


PersonTrackerV2 sẵn sàng — ngưỡng cảnh báo: 10s


BƯỚC 6: INFERENCE VIDEO

---
## Cập nhật: process_video_v2


In [8]:
# ============================================================
# 🎬 PROCESS VIDEO — SỬ DỤNG PersonTrackerV2 ĐÃ CẬP NHẬT
# ============================================================
from facenet_pytorch import MTCNN
import cv2, torch, time
from PIL import Image
from torchvision import transforms

def process_video_v2(model, video_path, output_path="output_video.mp4",
                     alert_seconds=10):
    """
    Xử lý video với tracking + đếm thời gian đeo khẩu trang.
    Cảnh báo khi đeo > alert_seconds giây.
    """
    PersonTrackerV2.ALERT_SECONDS = alert_seconds

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model  = model.to(device).eval()

    mtcnn   = MTCNN(keep_all=True, device=device)
    tracker = PersonTrackerV2()

    cap    = cv2.VideoCapture(video_path)
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    fps    = int(cap.get(cv2.CAP_PROP_FPS)) or 25
    W      = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H      = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    out    = cv2.VideoWriter(output_path, fourcc, fps, (W, H))

    transform = transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    class_names = ["mask_weared_incorrect", "with_mask", "without_mask"]

    t0, frame_no = time.time(), 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        frame_no   += 1
        now         = time.time() - t0
        img_rgb     = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        boxes, probs = mtcnn.detect(img_rgb)

        detections = []
        if boxes is not None:
            for box, prob in zip(boxes, probs):
                if prob is None or prob < 0.9:
                    continue
                x1, y1, x2, y2 = map(int, box)
                face = img_rgb[max(0,y1):y2, max(0,x1):x2]
                if face.size == 0:
                    continue
                face_t = transform(Image.fromarray(face)).unsqueeze(0).to(device)
                with torch.no_grad():
                    out_t  = model(face_t)
                    cls_id = out_t.argmax(dim=1).item()
                    conf   = out_t.softmax(dim=1)[0][cls_id].item()
                detections.append(([x1, y1, x2, y2], conf, cls_id))

        results = tracker.update(detections, img_rgb, now)
        frame   = PersonTrackerV2.draw(frame, results, class_names)

        # Overlay FPS & thời gian
        cv2.putText(frame, f"Frame {frame_no} | {now:.1f}s",
                    (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (200, 200, 200), 1)
        out.write(frame)

        if frame_no % 60 == 0:
            print(f"  ✔ {frame_no} frames — {now:.1f}s elapsed")

    cap.release(); out.release()
    print(f"\n✅ Video đã lưu: {output_path}")

# Sử dụng:
# model.load_state_dict(torch.load("/content/drive/MyDrive/best_mask_model.pth"))
# process_video_v2(model, "input.mp4", "output.mp4", alert_seconds=10)
print("✅ process_video_v2() sẵn sàng sử dụng")


✅ process_video_v2() sẵn sàng sử dụng


In [31]:
# CHuẩn bị và cân bằng dữ liệu
import os
import random
from collections import defaultdict, Counter
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, Subset
from torchvision import transforms

# ===== Hàm giới hạn số lượng mỗi class =====
def limit_dataset_per_class(dataset, max_per_class=1750):
    class_indices = defaultdict(list)

    # Gom index theo class
    for idx, label in enumerate(dataset.targets):
        class_indices[label].append(idx)

    selected_indices = []

    for label, indices in class_indices.items():
        if len(indices) > max_per_class:
            indices = random.sample(indices, max_per_class)
        selected_indices.extend(indices)

    random.shuffle(selected_indices)

    return Subset(dataset, selected_indices)

# ===== Hàm chuẩn bị dữ liệu =====
def prepare_data(data_dir='/content/dataset/Dataset_mask', batch_size=32, balance_train=True):

    # ============================================================
# TRAIN TRANSFORM
# ============================================================
    train_transform = transforms.Compose([

    transforms.Resize((128, 128)),

    transforms.RandomHorizontalFlip(p=0.5),

    transforms.RandomRotation(10),

    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

# ============================================================
# VAL / TEST TRANSFORM
# ============================================================
    test_transform = transforms.Compose([

    transforms.Resize((128, 128)),

    transforms.ToTensor(),

    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

    # Load dataset
    train_set = ImageFolder(
    os.path.join(data_dir, 'train'),
    transform=train_transform
)

    val_set = ImageFolder(
    os.path.join(data_dir, 'val'),
    transform=test_transform
)

    test_set = ImageFolder(
    os.path.join(data_dir, 'test'),
    transform=test_transform
)

    print(" Trước khi cân bằng:")
    print("Train:", Counter(train_set.targets))
    print("Val:", Counter(val_set.targets))
    print("Test:", Counter(test_set.targets))

    #  Cân bằng train
    if balance_train:
        train_set = limit_dataset_per_class(train_set, max_per_class=1750)

        # Lấy lại labels sau khi subset
        subset_labels = [train_set.dataset.targets[i] for i in train_set.indices]
        print("\n Sau khi cân bằng Train:")
        print("Train:", Counter(subset_labels))

    # DataLoader
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False)

    return train_loader, val_loader, test_loader

In [32]:
from collections import Counter
import torch

def get_labels(dataset):
    if isinstance(dataset, torch.utils.data.Subset):
        return [dataset.dataset.targets[i] for i in dataset.indices]
    return dataset.targets

train_loader, val_loader, test_loader = prepare_data()

# lấy dataset từ loader
train_set = train_loader.dataset
val_set = val_loader.dataset
test_set = test_loader.dataset

print("Train:", Counter(get_labels(train_set)))
print("Val:", Counter(get_labels(val_set)))
print("Test:", Counter(get_labels(test_set)))

 Trước khi cân bằng:
Train: Counter({0: 1750, 1: 1750, 2: 1750})
Val: Counter({0: 375, 1: 375, 2: 375})
Test: Counter({0: 375, 1: 375, 2: 375})

 Sau khi cân bằng Train:
Train: Counter({1: 1750, 2: 1750, 0: 1750})
Train: Counter({1: 1750, 2: 1750, 0: 1750})
Val: Counter({0: 375, 1: 375, 2: 375})
Test: Counter({0: 375, 1: 375, 2: 375})


BƯỚC 7: MAIN - CHẠY TRÊN COLAB

In [ ]:
def main():

    print("="*60)
    print("MASK DETECTION - OPTIMIZER COMPARISON")
    print("="*60)

    # =========================
    # DATA
    # =========================
    train_loader, val_loader, test_loader = prepare_data()

    # ========================================================
    # TRAIN ADAM
    # ========================================================
    print("\nTRAINING WITH ADAM\n")

    model_adam = MaskDetectionTransfer(num_classes=3)

    model_adam, history_adam = train_model(
        model_adam,
        train_loader,
        val_loader,
        num_epochs=10,
        optimizer_name="adam"
    )

    # ========================================================
    # TRAIN SGD
    # ========================================================
    print("\nTRAINING WITH SGD\n")

    model_sgd = MaskDetectionTransfer(num_classes=3)

    model_sgd, history_sgd = train_model(
        model_sgd,
        train_loader,
        val_loader,
        num_epochs=10,
        optimizer_name="sgd"
    )

    # ========================================================
    # LOAD BEST MODEL (optional nhưng nên có)
    # ========================================================
    model_adam.load_state_dict(
        torch.load("/content/drive/MyDrive/best_model_adam.pth")
    )

    model_sgd.load_state_dict(
        torch.load("/content/drive/MyDrive/best_model_sgd.pth")
    )

    # ========================================================
    # EVALUATE
    # ========================================================
    print("\nEVALUATE ADAM")
    evaluate_model(model_adam, test_loader)

    print("\nEVALUATE SGD")
    evaluate_model(model_sgd, test_loader)

    # ========================================================
    # PLOT COMPARISON LOSS / ACC
    # ========================================================
    plot_comparison(history_adam, history_sgd)


    print("\nHOÀN THÀNH SO SÁNH OPTIMIZER!")

if __name__ == "__main__":
      main()

MASK DETECTION - OPTIMIZER COMPARISON
 Trước khi cân bằng:
Train: Counter({0: 1750, 1: 1750, 2: 1750})
Val: Counter({0: 375, 1: 375, 2: 375})
Test: Counter({0: 375, 1: 375, 2: 375})

 Sau khi cân bằng Train:
Train: Counter({1: 1750, 0: 1750, 2: 1750})

TRAINING WITH ADAM


Đang sử dụng optimizer: ADAM


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)



Epoch 1/10
165/165 ━━━━━━━━━━━━━━━━━━━━ 60s - accuracy: 0.8457 - loss: 0.4065 - val_accuracy: 0.8267 - val_loss: 0.5135
Đã lưu model tốt nhất!

Epoch 2/10
165/165 ━━━━━━━━━━━━━━━━━━━━ 56s - accuracy: 0.8945 - loss: 0.2726 - val_accuracy: 0.8444 - val_loss: 0.4270
Đã lưu model tốt nhất!

Epoch 3/10
165/165 ━━━━━━━━━━━━━━━━━━━━ 56s - accuracy: 0.8979 - loss: 0.2658 - val_accuracy: 0.8347 - val_loss: 0.4489
Không cải thiện (1/5)

Epoch 4/10
165/165 ━━━━━━━━━━━━━━━━━━━━ 56s - accuracy: 0.9156 - loss: 0.2456 - val_accuracy: 0.8364 - val_loss: 0.4106
Đã lưu model tốt nhất!

Epoch 5/10
165/165 ━━━━━━━━━━━━━━━━━━━━ 56s - accuracy: 0.9135 - loss: 0.2382 - val_accuracy: 0.8498 - val_loss: 0.3671
Đã lưu model tốt nhất!

Epoch 6/10
165/165 ━━━━━━━━━━━━━━━━━━━━ 55s - accuracy: 0.9286 - loss: 0.1991 - val_accuracy: 0.8489 - val_loss: 0.3675
Không cải thiện (1/5)

Epoch 7/10
165/165 ━━━━━━━━━━━━━━━━━━━━ 53s - accuracy: 0.9280 - loss: 0.2016 - val_accuracy: 0.8507 - val_loss: 0.4060
Không cải thiện (

In [11]:
# VẼ các sơ đồ
# So sánh 2 hàm tối ưu
import matplotlib.pyplot as plt

def plot_comparison(history_adam, history_sgd):

    epochs = range(1, len(history_adam["train_loss"]) + 1)

    plt.figure(figsize=(12, 5))

    # =========================
    # LOSS
    # =========================
    plt.subplot(1, 2, 1)

    plt.plot(epochs, history_adam["val_loss"], label="Adam", linewidth=2)
    plt.plot(epochs, history_sgd["val_loss"], label="SGD", linewidth=2)

    plt.title("Validation Loss Comparison")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(alpha=0.3)

    # =========================
    # ACCURACY
    # =========================
    plt.subplot(1, 2, 2)

    plt.plot(epochs, history_adam["val_acc"], label="Adam", linewidth=2)
    plt.plot(epochs, history_sgd["val_acc"], label="SGD", linewidth=2)

    plt.title("Validation Accuracy Comparison")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy (%)")
    plt.legend()
    plt.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

---
## Sơ đồ Loss & Accuracy


In [12]:
# ============================================================
#  VẼ SƠ ĐỒ LOSS & ACCURACY (STYLE BASIC)
# ============================================================
import matplotlib.pyplot as plt

def plot_training_history(history):
    """
    Vẽ biểu đồ Loss và Accuracy theo từng epoch.
    history: dict gồm train_loss, val_loss, train_acc, val_acc
    """

    epochs = range(1, len(history["train_loss"]) + 1)

    # Tạo figure
    plt.figure(figsize=(12, 5))

    # =====================================================
    # LOSS
    # =====================================================
    plt.subplot(1, 2, 1)

    plt.plot(
        epochs,
        history["train_loss"],
        label="Train Loss",
        marker="o"
    )

    plt.plot(
        epochs,
        history["val_loss"],
        label="Validation Loss",
        marker="s",
        linestyle="--"
    )

    plt.title("Loss per Epoch")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True)

    # =====================================================
    # ACCURACY
    # =====================================================
    plt.subplot(1, 2, 2)

    plt.plot(
        epochs,
        history["train_acc"],
        label="Train Accuracy",
        marker="o"
    )

    plt.plot(
        epochs,
        history["val_acc"],
        label="Validation Accuracy",
        marker="s",
        linestyle="--"
    )

    plt.title("Accuracy per Epoch")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy (%)")
    plt.legend()
    plt.grid(True)

    # =====================================================
    # HIỂN THỊ
    # =====================================================
    plt.tight_layout()

    # Lưu ảnh
    plt.savefig("training_history.png", dpi=120)

    plt.show()

    print("Đã lưu biểu đồ: training_history.png")


# ============================================================
# DEMO TEST
# ============================================================
# demo_history = {
#     "train_loss": [1.1, 0.85, 0.65, 0.50, 0.40, 0.33, 0.27, 0.23, 0.20, 0.18],
#     "val_loss":   [1.0, 0.80, 0.62, 0.52, 0.45, 0.42, 0.40, 0.39, 0.38, 0.37],

#     "train_acc":  [45, 62, 74, 82, 87, 90, 92, 94, 95, 96],
#     "val_acc":    [48, 60, 72, 79, 83, 84, 85, 86, 86.5, 87],
# }

# plot_training_history(demo_history)

---
## 🔍 Bổ sung: Grad-CAM Visualization


In [ ]:
# ============================================================
# 🔍 BƯỚC BỔ SUNG: GRAD-CAM — TRỰC QUAN HÓA MODEL NHÌN GÌ
# ============================================================
import torch
import torch.nn.functional as F
import numpy as np
import cv2
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms

class GradCAM:
    """
    Grad-CAM: tính gradient của class score
    so với activation map của conv layer cuối.
    Cho thấy model "chú ý" vào vùng nào của ảnh.
    """
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        self._register_hooks()

    def _register_hooks(self):
        def forward_hook(module, input, output):
            self.activations = output.detach()

        def backward_hook(module, grad_input, grad_output):
            self.gradients = grad_output[0].detach()

        self.target_layer.register_forward_hook(forward_hook)
        self.target_layer.register_full_backward_hook(backward_hook)

    def generate(self, input_tensor, target_class=None):
        self.model.eval()
        output = self.model(input_tensor)                    # forward

        if target_class is None:
            target_class = output.argmax(dim=1).item()

        # Backward với class score
        self.model.zero_grad()
        one_hot = torch.zeros_like(output)
        one_hot[0][target_class] = 1
        output.backward(gradient=one_hot, retain_graph=True)

        # Grad-CAM formula: alpha * activation
        grads   = self.gradients[0]          # (C, H, W)
        acts    = self.activations[0]        # (C, H, W)
        weights = grads.mean(dim=(1, 2))     # global average pooling

        cam = (weights[:, None, None] * acts).sum(dim=0)
        cam = F.relu(cam)
        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)
        return cam.cpu().numpy(), target_class, output.softmax(dim=1)[0].detach().cpu().numpy()


def overlay_gradcam(img_np, cam, alpha=0.45):
    """Overlay heatmap lên ảnh gốc."""
    h, w = img_np.shape[:2]
    cam_resized = cv2.resize(cam, (w, h))
    heatmap = cv2.applyColorMap((cam_resized * 255).astype(np.uint8), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    overlay = (img_np * (1 - alpha) + heatmap * alpha).astype(np.uint8)
    return overlay, cam_resized


def visualize_gradcam_batch(model, image_paths, class_names, transform):
    """
    Vẽ lưới ảnh gốc | Grad-CAM overlay | bar chart confidence
    cho nhiều ảnh cùng lúc.
    """
    # Lấy conv layer cuối của model (conv4)
    target_layer = model.conv4
    grad_cam = GradCAM(model, target_layer)

    n = len(image_paths)
    fig, axes = plt.subplots(n, 3, figsize=(12, 4 * n))
    fig.patch.set_facecolor("#0d1117")
    if n == 1:
        axes = [axes]

    col_titles = ["Ảnh gốc", "Grad-CAM Heatmap", "Confidence Score"]
    for j, title in enumerate(col_titles):
        axes[0][j].set_title(title, color="#8b949e", fontsize=11, fontweight="bold", pad=8)

    label_colors = {
        "mask_weared_incorrect": "#ffd93d",
        "with_mask":             "#6bcb77",
        "without_mask":          "#ff6b6b",
    }

    for i, img_path in enumerate(image_paths):
        img = Image.open(img_path).convert("RGB").resize((128, 128))
        img_np = np.array(img)

        input_tensor = transform(img).unsqueeze(0)
        cam, pred_class, probs = grad_cam.generate(input_tensor)

        overlay, _ = overlay_gradcam(img_np, cam)
        pred_name   = class_names[pred_class]
        pred_color  = label_colors.get(pred_name, "white")

        # Col 0: Ảnh gốc
        ax0 = axes[i][0]
        ax0.imshow(img_np)
        ax0.set_facecolor("#0d1117")
        ax0.set_ylabel(f"Dự đoán:\n{pred_name}\n({probs[pred_class]:.1%})",
                       color=pred_color, fontsize=9, fontweight="bold",
                       rotation=0, labelpad=80, va="center")
        ax0.axis("off")

        # Col 1: Grad-CAM overlay
        ax1 = axes[i][1]
        ax1.imshow(overlay)
        ax1.set_facecolor("#0d1117")
        ax1.axis("off")

        # Vẽ vùng có attention cao nhất (contour)
        mask = (cv2.resize(cam, (128, 128)) > 0.6).astype(np.uint8) * 255
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        overlay_annotated = overlay.copy()
        cv2.drawContours(overlay_annotated, contours, -1, (255, 255, 0), 2)
        ax1.imshow(overlay_annotated)

        # Col 2: Confidence bar
        ax2 = axes[i][2]
        ax2.set_facecolor("#161b22")
        colors_bar = [label_colors.get(c, "white") for c in class_names]
        bars = ax2.barh(class_names, probs * 100, color=colors_bar, edgecolor="#30363d")
        for bar, prob in zip(bars, probs):
            ax2.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
                     f"{prob:.1%}", va="center", color="white", fontsize=9)
        ax2.set_xlim(0, 115)
        ax2.set_xlabel("Confidence (%)", color="#8b949e")
        ax2.tick_params(colors="white")
        for sp in ax2.spines.values(): sp.set_color("#30363d")
        ax2.axvline(x=50, color="white", linestyle=":", alpha=0.3)

        # Highlight predicted class
        bars[pred_class].set_linewidth(2.5)
        bars[pred_class].set_edgecolor("white")

    fig.suptitle("🔍 Grad-CAM — Vùng model chú ý khi phán đoán",
                 color="white", fontsize=14, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.savefig("gradcam_visualization.png", dpi=150, bbox_inches="tight", facecolor="#0d1117")
    plt.show()
    print("✅ Đã lưu: gradcam_visualization.png")


# ── CÁCH SỬ DỤNG ──────────────────────────────────────────────
# Sau khi train xong:
#
# transform = transforms.Compose([
#     transforms.Resize((128, 128)),
#     transforms.ToTensor(),
#     transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
# ])
# class_names = ["mask_weared_incorrect", "with_mask", "without_mask"]
#
# image_paths = [
#     "/content/dataset/test/with_mask/sample1.jpg",
#     "/content/dataset/test/without_mask/sample2.jpg",
# ]
# visualize_gradcam_batch(model, image_paths, class_names, transform)
print("GradCAM class đã sẵn sàng — gọi visualize_gradcam_batch() sau khi train")


GradCAM class đã sẵn sàng — gọi visualize_gradcam_batch() sau khi train
